## Set up

### libraries

In [19]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict


In [20]:
# ==================== CONFIGURATION MAPPING ====================
# Easy customization: Change what each number in the config represents
# Position 3 (3rd dot-separated value): Turbulence models or other classifications
# Position 4 (4th dot-separated value): Grid/Adaptation status or other classifications

CONFIG_MAPPING = {
    'position_3': {
        '1': 'SST',
        '2': 'RNG',
        '3': 'RSM'
        # Add more as needed: '4': 'NewModel', '5': 'AnotherModel', etc.
    },
    'position_4': {
        '1': 'Baseline',
        '2': 'Adapted',
        '3': 'Adapted With Grid Fins',
        '4': 'Grid Fins'
        # Add more as needed: '5': 'CustomConfig', '6': 'AnotherStatus', etc.
    }
}

print("✓ Configuration mapping loaded")
print(f"  Position 3 options: {list(CONFIG_MAPPING['position_3'].values())}")
print(f"  Position 4 options: {list(CONFIG_MAPPING['position_4'].values())}")


✓ Configuration mapping loaded
  Position 3 options: ['SST', 'RNG', 'RSM']
  Position 4 options: ['Baseline', 'Adapted', 'Adapted With Grid Fins', 'Grid Fins']


### Data Extraction

In [21]:
def load_lift_drag_data(root_dir):
    data_by_config_aoa = defaultdict(lambda: {'lift': [], 'drag': [], 'turbulence_model': '', 'adapted': '', 'grid_fins': ''})
    
    for dirpath, _, filenames in os.walk(root_dir):
        # Find lift and drag files (with pattern like lift_force_AoA_X_3_1.txt)
        lift_files = [f for f in filenames if 'lift_force' in f and f.endswith('.txt')]
        drag_files = [f for f in filenames if 'drag_force' in f and f.endswith('.txt')]
        
        if lift_files and drag_files:
            # Extract configuration (e.g., 4.3.1.2) from path
            parts = dirpath.split(os.sep)
            config = None
            aoa = None
            
            # Find config number (4.X.X.X format)
            for part in parts:
                if part[0].isdigit() and len(part) == 7 and part.count('.') == 3:
                    config = part
                    break
            
            # Find AoA value from folder name (e.g., AoA_10)
            for part in parts:
                if part.startswith('AoA_'):
                    aoa = part
                    break
            
            if config and aoa:
                # Extract configuration values
                config_parts = config.split('.')
                third_num = config_parts[2] if len(config_parts) > 2 else "unknown"
                last_num = config_parts[3] if len(config_parts) > 3 else "unknown"
                
                # Get turbulence model from CONFIG_MAPPING
                turbulence_model = CONFIG_MAPPING['position_3'].get(third_num, "Unknown")
                
                # Get status from CONFIG_MAPPING for position_4
                # Special handling: position_4 uses custom logic for adapted/grid_fins
                # Check the mapping and set flags accordingly
                position_4_value = CONFIG_MAPPING['position_4'].get(last_num, "Unknown")
                
                if position_4_value == "Baseline":
                    adapted = "No"
                    grid_fins = "No"
                elif position_4_value == "Adapted":
                    adapted = "Yes"
                    grid_fins = "No"
                elif position_4_value == "Grid_Fins":
                    adapted = "No"
                    grid_fins = "Yes"
                else:
                    adapted = "Unknown"
                    grid_fins = "Unknown"
                
                # Read all lift files
                for lift_file in lift_files:
                    lift_path = os.path.join(dirpath, lift_file)
                    with open(lift_path) as f:
                        lift_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    lift_data.append(value)
                                except ValueError:
                                    continue
                        data_by_config_aoa[(config, aoa)]['lift'].extend(lift_data)
                
                # Read all drag files
                for drag_file in drag_files:
                    drag_path = os.path.join(dirpath, drag_file)
                    with open(drag_path) as f:
                        drag_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    drag_data.append(value)
                                except ValueError:
                                    continue
                        data_by_config_aoa[(config, aoa)]['drag'].extend(drag_data)
                
                # Store metadata
                data_by_config_aoa[(config, aoa)]['turbulence_model'] = turbulence_model
                data_by_config_aoa[(config, aoa)]['adapted'] = adapted
                data_by_config_aoa[(config, aoa)]['grid_fins'] = grid_fins
    
    return data_by_config_aoa


In [22]:
def compute_statistics(data):
    mean_val = np.mean(data)
    std_dev = np.std(data)
    # Calculate Coefficient of Variation (COV) as percentage
    cov = (std_dev / mean_val * 100) if mean_val != 0 else 0
    return mean_val, cov

## Enter Values Here

In [30]:
# ==================== CONFIGURATION ====================
# User parameters - modify these as needed

base_path = r"C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3"
output_dir = r"C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3"
num_iterations = 150  # Number of latest iterations to use for statistics

# ==================== PROCESSING ====================

# Load all data
all_data = load_lift_drag_data(base_path)

# Create output directories
os.makedirs(output_dir, exist_ok=True)
raw_data_dir = os.path.join(output_dir, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

# Sort AoA values numerically (5, 10, 15, 20)
def extract_aoa_number(aoa_str):
    return int(aoa_str.split('_')[1])

# Export each dataset with a unique name based on config and AOA
for (config, aoa), data in all_data.items():
    filename_base = f"{config}_{aoa}"
    
    # Export lift data to raw_data folder
    lift_output = os.path.join(raw_data_dir, f"{filename_base}_lift.txt")
    with open(lift_output, 'w') as f:
        for value in data['lift']:
            f.write(f"{value}\n")
    
    # Export drag data to raw_data folder
    drag_output = os.path.join(raw_data_dir, f"{filename_base}_drag.txt")
    with open(drag_output, 'w') as f:
        for value in data['drag']:
            f.write(f"{value}\n")
    
    print(f"Exported: {filename_base}")

# Create summary statistics file using specified number of iterations
summary_file = os.path.join(output_dir, "SUMMARY_Statistics.txt")

with open(summary_file, 'w') as f:
    f.write("=" * 100 + "\n")
    f.write(f"SUMMARY STATISTICS - Last {num_iterations} Iterations\n")
    f.write("=" * 100 + "\n\n")
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations (or all if less than N)
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        
        # Write to file
        f.write(f"Configuration: {config}\n")
        f.write(f"Turbulence Model: {data['turbulence_model']}\n")
        f.write(f"Adapted: {data['adapted']}\n")
        f.write(f"Grid Fins: {data['grid_fins']}\n")
        f.write(f"Angle of Attack: {aoa}\n")
        f.write(f"Iterations used: {len(lift_last_n)}\n\n")
        f.write(f"LIFT:\n")
        f.write(f"  Average: {lift_mean:12.6f}\n")
        f.write(f"  COV (%): {lift_cov:12.6f}\n\n")
        f.write(f"DRAG:\n")
        f.write(f"  Average: {drag_mean:12.6f}\n")
        f.write(f"  COV (%): {drag_cov:12.6f}\n")
        f.write("-" * 100 + "\n\n")

print(f"\n✓ Raw data exported to: {raw_data_dir}")
print(f"✓ Summary statistics saved to: {summary_file}")

Exported: 4.3.1.2_AoA_10
Exported: 4.3.1.2_AoA_15
Exported: 4.3.1.2_AoA_20
Exported: 4.3.1.2_AoA_5
Exported: 4.3.1.3_AoA_10
Exported: 4.3.1.3_AoA_15
Exported: 4.3.1.3_AoA_20
Exported: 4.3.1.3_AoA_5
Exported: 4.3.1.4_AoA_10
Exported: 4.3.1.4_AoA_15
Exported: 4.3.1.4_AoA_20
Exported: 4.3.1.4_AoA_5
Exported: 4.3.2.1_AoA_10
Exported: 4.3.2.1_AoA_15
Exported: 4.3.2.1_AoA_20
Exported: 4.3.2.1_AoA_5
Exported: 4.3.3.1_AoA_10
Exported: 4.3.3.1_AoA_15
Exported: 4.3.3.1_AoA_20
Exported: 4.3.3.1_AoA_5

✓ Raw data exported to: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\raw_data
✓ Summary statistics saved to: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\SUMMARY_Statistics.txt


In [24]:
# ==================== CREATE BAR GRAPHS ====================

graphs_dir = os.path.join(output_dir, "graphs")
os.makedirs(graphs_dir, exist_ok=True)

# Group data by AoA and turbulence model
aoa_groups = defaultdict(lambda: {'SST': {'lift': [], 'drag': []}, 'RNG': {'lift': [], 'drag': []}, 'RMS': {'lift': [], 'drag': []}})

for (config, aoa), data in all_data.items():
    # Skip if adapted=Yes or grid_fins=Yes
    if data['adapted'] == "Yes" or data['grid_fins'] == "Yes":
        continue
    
    turbulence_model = data['turbulence_model']
    
    # Get last N iterations for graphing
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate means
    lift_mean = np.mean(lift_last_n) if lift_last_n else 0
    drag_mean = np.mean(drag_last_n) if drag_last_n else 0
    
    if turbulence_model in aoa_groups[aoa]:
        aoa_groups[aoa][turbulence_model]['lift'].append(lift_mean)
        aoa_groups[aoa][turbulence_model]['drag'].append(drag_mean)

# Create bar graphs for each AoA
for aoa in sorted(aoa_groups.keys(), key=extract_aoa_number):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Prepare data for plotting
    models = ['SST', 'RNG', 'RMS']
    lift_means = []
    drag_means = []
    
    for model in models:
        if aoa_groups[aoa][model]['lift']:
            lift_means.append(np.mean(aoa_groups[aoa][model]['lift']))
        else:
            lift_means.append(0)
        
        if aoa_groups[aoa][model]['drag']:
            drag_means.append(np.mean(aoa_groups[aoa][model]['drag']))
        else:
            drag_means.append(0)
    
    # Create bar charts
    x = np.arange(len(models))
    width = 0.6
    
    ax1.bar(x, lift_means, width, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black', linewidth=1.5)
    ax1.set_title(f'Lift Comparison - {aoa}', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Lift Mean Value', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(models, fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(lift_means):
        ax1.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax2.bar(x, drag_means, width, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black', linewidth=1.5)
    ax2.set_title(f'Drag Comparison - {aoa}', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Drag Mean Value', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(models, fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(drag_means):
        ax2.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    # Save graph
    aoa_num = extract_aoa_number(aoa)
    graph_file = os.path.join(graphs_dir, f"turbulence_comparison_AoA_{aoa_num}.png")
    plt.savefig(graph_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Created bar graph for {aoa}: {graph_file}")

print(f"\n✓ All bar graphs saved to: {graphs_dir}")
print(f"✓ Note: Graphs exclude data with Adapted=Yes or Grid Fins=Yes")


Created bar graph for AoA_5: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_5.png
Created bar graph for AoA_10: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_10.png
Created bar graph for AoA_10: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_10.png
Created bar graph for AoA_15: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_15.png
Created bar graph for AoA_15: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_15.png
Created bar graph for AoA_20: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_20.png

✓ All bar graphs saved to: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs
✓ Note: Graphs exclude data with Adapted=Yes or Grid Fins=Yes
Created bar graph for AoA_20: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_compa

In [25]:
# ==================== CREATE BAR GRAPHS - GRID FINS ON ====================

# Group data by AoA (only Grid Fins = Yes)
grid_fins_data = defaultdict(lambda: {'lift': [], 'drag': []})

for (config, aoa), data in all_data.items():
    # Only include if grid_fins=Yes (last number = 4)
    if data['grid_fins'] != "Yes":
        continue
    
    # Get last N iterations for graphing
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate means
    lift_mean = np.mean(lift_last_n) if lift_last_n else 0
    drag_mean = np.mean(drag_last_n) if drag_last_n else 0
    
    grid_fins_data[aoa]['lift'].append(lift_mean)
    grid_fins_data[aoa]['drag'].append(drag_mean)

# Create one large bar graph comparing all AoAs (Grid Fins)
if grid_fins_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Prepare data for plotting
    aoas = sorted(grid_fins_data.keys(), key=extract_aoa_number)
    lift_means = []
    drag_means = []
    
    for aoa in aoas:
        if grid_fins_data[aoa]['lift']:
            lift_means.append(np.mean(grid_fins_data[aoa]['lift']))
        else:
            lift_means.append(0)
        
        if grid_fins_data[aoa]['drag']:
            drag_means.append(np.mean(grid_fins_data[aoa]['drag']))
        else:
            drag_means.append(0)
    
    # Create bar charts
    x = np.arange(len(aoas))
    width = 0.6
    
    ax1.bar(x, lift_means, width, color='#1f77b4', edgecolor='black', linewidth=1.5)
    ax1.set_title('Lift Comparison - Grid Fins ON (All AoA)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Lift Mean Value', fontsize=12)
    ax1.set_xlabel('Angle of Attack', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(aoas, fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(lift_means):
        ax1.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax2.bar(x, drag_means, width, color='#ff7f0e', edgecolor='black', linewidth=1.5)
    ax2.set_title('Drag Comparison - Grid Fins ON (All AoA)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Drag Mean Value', fontsize=12)
    ax2.set_xlabel('Angle of Attack', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(aoas, fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(drag_means):
        ax2.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    # Save graph
    graph_file = os.path.join(graphs_dir, f"grid_fins_all_aoa_comparison.png")
    plt.savefig(graph_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Created grid fins comparison graph: {graph_file}")

print(f"✓ Grid fins bar graph saved to: {graphs_dir}")

# ==================== CREATE BAR GRAPHS - ADAPTED ====================

# Group data by AoA (only Adapted = Yes)
adapted_data = defaultdict(lambda: {'lift': [], 'drag': []})

for (config, aoa), data in all_data.items():
    # Only include if adapted=Yes (last number = 3)
    if data['adapted'] != "Yes":
        continue
    
    # Get last N iterations for graphing
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate means
    lift_mean = np.mean(lift_last_n) if lift_last_n else 0
    drag_mean = np.mean(drag_last_n) if drag_last_n else 0
    
    adapted_data[aoa]['lift'].append(lift_mean)
    adapted_data[aoa]['drag'].append(drag_mean)

# Create one large bar graph comparing all AoAs (Adapted)
if adapted_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Prepare data for plotting
    aoas = sorted(adapted_data.keys(), key=extract_aoa_number)
    lift_means = []
    drag_means = []
    
    for aoa in aoas:
        if adapted_data[aoa]['lift']:
            lift_means.append(np.mean(adapted_data[aoa]['lift']))
        else:
            lift_means.append(0)
        
        if adapted_data[aoa]['drag']:
            drag_means.append(np.mean(adapted_data[aoa]['drag']))
        else:
            drag_means.append(0)
    
    # Create bar charts
    x = np.arange(len(aoas))
    width = 0.6
    
    ax1.bar(x, lift_means, width, color='#2ca02c', edgecolor='black', linewidth=1.5)
    ax1.set_title('Lift Comparison - Adapted (All AoA)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Lift Mean Value', fontsize=12)
    ax1.set_xlabel('Angle of Attack', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(aoas, fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(lift_means):
        ax1.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax2.bar(x, drag_means, width, color='#d62728', edgecolor='black', linewidth=1.5)
    ax2.set_title('Drag Comparison - Adapted (All AoA)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Drag Mean Value', fontsize=12)
    ax2.set_xlabel('Angle of Attack', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(aoas, fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(drag_means):
        ax2.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    # Save graph
    graph_file = os.path.join(graphs_dir, f"adapted_all_aoa_comparison.png")
    plt.savefig(graph_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Created adapted comparison graph: {graph_file}")

print(f"✓ Adapted bar graph saved to: {graphs_dir}")


✓ Grid fins bar graph saved to: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs
Created adapted comparison graph: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\adapted_all_aoa_comparison.png
✓ Adapted bar graph saved to: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs
Created adapted comparison graph: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs\adapted_all_aoa_comparison.png
✓ Adapted bar graph saved to: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\graphs


In [26]:
# ==================== CREATE EXCEL TABLES ====================

from openpyxl.utils import get_column_letter

# Create Excel file with all data
excel_file = os.path.join(output_dir, "SUMMARY_Statistics.xlsx")

# Recreate config_groups for reference
config_groups = defaultdict(lambda: {'lift': {}, 'drag': {}})
for (config, aoa), data in all_data.items():
    config_parts = config.split('.')
    third_num = config_parts[2] if len(config_parts) > 2 else "unknown"
    
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    config_groups[third_num]['lift'][aoa] = lift_last_n
    config_groups[third_num]['drag'][aoa] = drag_last_n

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Create a summary sheet with all configurations
    summary_data = []
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        
        summary_data.append({
            'Configuration': config,
            'Turbulence Model': data['turbulence_model'],
            'Adapted': data['adapted'],
            'Grid Fins': data['grid_fins'],
            'Angle of Attack': aoa,
            'Iterations Used': len(lift_last_n),
            'Lift Mean (N)': f"{lift_mean:.6f}",
            'Lift COV (%)': f"{lift_cov:.6f}",
            'Drag Mean (N)': f"{drag_mean:.6f}",
            'Drag COV (%)': f"{drag_cov:.6f}"
        })
    
    # Write summary to main sheet
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Create separate sheets for each configuration type (by third number)
    for third_num in sorted(config_groups.keys()):
        config_data = []
        
        # Sort AoA numerically within each config type
        sorted_aoas = sorted(config_groups[third_num]['lift'].keys(), key=extract_aoa_number)
        
        for aoa in sorted_aoas:
            lift_data = config_groups[third_num]['lift'][aoa]
            drag_data = config_groups[third_num]['drag'][aoa]
            
            # Get turbulence model info (should be same for all in this group)
            turbulence_model = ""
            adapted = ""
            grid_fins = ""
            for (config, config_aoa), d in all_data.items():
                if config_aoa == aoa:
                    config_parts = config.split('.')
                    if config_parts[2] == third_num:
                        turbulence_model = d['turbulence_model']
                        adapted = d['adapted']
                        grid_fins = d['grid_fins']
                        break
            
            # Calculate COV for lift and drag
            lift_mean = np.mean(lift_data)
            lift_std = np.std(lift_data)
            lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
            
            drag_mean = np.mean(drag_data)
            drag_std = np.std(drag_data)
            drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
            
            config_data.append({
                'Angle of Attack': aoa,
                'Turbulence Model': turbulence_model,
                'Adapted': adapted,
                'Grid Fins': grid_fins,
                'Lift Mean (N)': f"{lift_mean:.6f}",
                'Lift COV (%)': f"{lift_cov:.6f}",
                'Drag Mean (N)': f"{drag_mean:.6f}",
                'Drag COV (%)': f"{drag_cov:.6f}",
                'Num Points': len(lift_data)
            })
        
        config_df = pd.DataFrame(config_data)
        
        # Get sheet name from CONFIG_MAPPING
        sheet_name = CONFIG_MAPPING['position_3'].get(third_num, f'Config_Type_{third_num}')
        
        config_df.to_excel(writer, sheet_name=sheet_name, index=False)

# Apply autofit to all sheets
from openpyxl import load_workbook
wb = load_workbook(excel_file)
for ws in wb.sheetnames:
    worksheet = wb[ws]
    
    # Autofit column widths
    for column in worksheet.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)  # Cap at 50 characters
        worksheet.column_dimensions[column_letter].width = adjusted_width
    
    # Autofit row heights
    for row in worksheet.iter_rows():
        max_height = 15  # Default minimum height
        for cell in row:
            if cell.value:
                cell_text = str(cell.value)
                # Estimate height based on number of lines
                line_count = cell_text.count('\n') + 1
                estimated_height = line_count * 15
                if estimated_height > max_height:
                    max_height = estimated_height
        worksheet.row_dimensions[row[0].row].height = max_height

wb.save(excel_file)

sheet_names = [CONFIG_MAPPING['position_3'].get(k, f'Config_Type_{k}') for k in sorted(config_groups.keys())]
print(f"✓ Excel file created: {excel_file}")
print(f"✓ Sheet names: Summary, {', '.join(sheet_names)}")
print(f"✓ Column widths and row heights auto-fitted")

✓ Excel file created: C:\Users\lukek\OneDrive\Documents\Test\2414_006_004.3\SUMMARY_Statistics.xlsx
✓ Sheet names: Summary, SST, RNG, RSM
✓ Column widths and row heights auto-fitted


In [29]:
# ==================== CREATE TURBULENCE MODEL COMPARISON TABLE ====================

import time

# Create a comparison table for turbulence models
# Try baseline first, then fall back to first available config for each model
turbulence_comparison = []

# Get all unique AoAs
all_aoas = set()
for (config, aoa), data in all_data.items():
    all_aoas.add(aoa)

# Sort AoAs numerically
sorted_aoas = sorted(all_aoas, key=extract_aoa_number)
# Dynamically determine which turbulence models are available
all_models = set()
for (config, aoa), data in all_data.items():
    all_models.add(data['turbulence_model'])

print(f"✓ Turbulence models in dataset: {sorted(all_models)}")

# For each AoA, get data for each turbulence model
for aoa in sorted_aoas:
    aoa_entry = {'Angle of Attack': aoa}
    
    # Collect lift and drag means for each model
    lift_means = {}
    drag_means = {}
    lift_covs = {}
    drag_covs = {}
    
    for model in sorted(all_models):
        model_lift_values = []
        model_drag_values = []
        
        # Find all configs with this AOA and turbulence model (any configuration)
        for (config, config_aoa), data in all_data.items():
            if config_aoa == aoa and data['turbulence_model'] == model:
                lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
                drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
                
                if lift_last_n:
                    model_lift_values.append(np.mean(lift_last_n))
                if drag_last_n:
                    model_drag_values.append(np.mean(drag_last_n))
        
        # Store averages for this model
        if model_lift_values:
            lift_means[model] = np.mean(model_lift_values)
            lift_all = []
            for (config, config_aoa), data in all_data.items():
                if config_aoa == aoa and data['turbulence_model'] == model:
                    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
                    lift_all.extend(lift_last_n)
            if lift_all:
                lift_std = np.std(lift_all)
                lift_mean = np.mean(lift_all)
                lift_covs[model] = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        else:
            lift_means[model] = None
            lift_covs[model] = None
        
        if model_drag_values:
            drag_means[model] = np.mean(model_drag_values)
            drag_all = []
            for (config, config_aoa), data in all_data.items():
                if config_aoa == aoa and data['turbulence_model'] == model:
                    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
                    drag_all.extend(drag_last_n)
            if drag_all:
                drag_std = np.std(drag_all)
                drag_mean = np.mean(drag_all)
                drag_covs[model] = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        else:
            drag_means[model] = None
            drag_covs[model] = None
    
    # Add lift values in format: Model - value (COV), Model - value (COV), etc.
    lift_str = ""
    for model in sorted(all_models):
        if lift_means[model] is not None and lift_covs[model] is not None:
            if lift_str:
                lift_str += " | "
            lift_str += f"{model}: {lift_means[model]:.4f} N ({lift_covs[model]:.2f}%)"
        else:
            if lift_str:
                lift_str += " | "
            lift_str += f"{model}: N/A"
    aoa_entry['Lift (N)'] = lift_str
    
    # Add drag values in format: Model - value (COV), Model - value (COV), etc.
    drag_str = ""
    for model in sorted(all_models):
        if drag_means[model] is not None and drag_covs[model] is not None:
            if drag_str:
                drag_str += " | "
            drag_str += f"{model}: {drag_means[model]:.4f} N ({drag_covs[model]:.2f}%)"
        else:
            if drag_str:
                drag_str += " | "
            drag_str += f"{model}: N/A"
    aoa_entry['Drag (N)'] = drag_str
    
    turbulence_comparison.append(aoa_entry)

# Add to existing Excel file
comparison_df = pd.DataFrame(turbulence_comparison)

# Write comparison sheet (close any open Excel processes first)
time.sleep(1)  # Wait for any file locks to clear

try:
    with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        comparison_df.to_excel(writer, sheet_name='Turbulence_Model_Comparison', index=False)
except PermissionError:
    print("Warning: Excel file is locked. Please close the file and re-run this cell.")
    with pd.ExcelWriter(excel_file.replace('.xlsx', '_temp.xlsx'), engine='openpyxl') as writer:
        comparison_df.to_excel(writer, sheet_name='Turbulence_Model_Comparison', index=False)

# Apply autofit to Turbulence_Model_Comparison sheet
time.sleep(0.5)

try:
    wb = load_workbook(excel_file)
    ws = wb['Turbulence_Model_Comparison']

    # Autofit column widths
    for column in ws.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 80)
        ws.column_dimensions[column_letter].width = adjusted_width

    # Autofit row heights
    for row in ws.iter_rows():
        max_height = 15
        for cell in row:
            if cell.value:
                cell_text = str(cell.value)
                line_count = cell_text.count('\n') + 1
                estimated_height = line_count * 15
                if estimated_height > max_height:
                    max_height = estimated_height
        ws.row_dimensions[row[0].row].height = max_height

    wb.save(excel_file)
    wb.close()
except PermissionError:
    print("Note: Could not update Excel formatting due to file lock.")

print(f"\n✓ Turbulence Model Comparison table added to Excel file")
print(f"✓ Models compared: {', '.join(sorted(all_models))}")
print(f"✓ Format: Model: value (COV%) | ... (easy to compare at a glance)")
print(f"✓ Note: Includes all available configurations for each model")
print(f"\nSample comparison:")
for row in turbulence_comparison:
    print(f"\n{row['Angle of Attack']}:")
    print(f"  Lift:  {row['Lift (N)']}")
    print(f"  Drag:  {row['Drag (N)']}")

✓ Turbulence models in baseline data: ['RNG', 'RSM']

✓ Turbulence Model Comparison table added to Excel file
✓ Models compared: RNG, RSM
✓ Format: Model: value (COV%) | ... (easy to compare at a glance)

Sample comparison:

AoA_5:
  Lift:  RNG: 66.8552 N (0.00%) | RSM: 64.5866 N (0.00%)
  Drag:  RNG: -3.7801 N (-0.00%) | RSM: -3.6998 N (-0.00%)

AoA_10:
  Lift:  RNG: 107.2339 N (0.00%) | RSM: 101.4589 N (0.00%)
  Drag:  RNG: -14.9987 N (-0.00%) | RSM: -14.1636 N (-0.00%)

AoA_15:
  Lift:  RNG: 90.7611 N (9.86%) | RSM: 85.4806 N (10.57%)
  Drag:  RNG: -15.7245 N (-18.97%) | RSM: -7.0584 N (-41.89%)

AoA_20:
  Lift:  RNG: 61.3692 N (8.13%) | RSM: 73.3639 N (5.61%)
  Drag:  RNG: -3.5327 N (-27.91%) | RSM: -1.9757 N (-56.35%)

✓ Turbulence Model Comparison table added to Excel file
✓ Models compared: RNG, RSM
✓ Format: Model: value (COV%) | ... (easy to compare at a glance)

Sample comparison:

AoA_5:
  Lift:  RNG: 66.8552 N (0.00%) | RSM: 64.5866 N (0.00%)
  Drag:  RNG: -3.7801 N (-0.00%